# Truncation Blind Spot -- Colab run notebook

Clones this repo and runs decoding arms from `config/arms.yaml` (12 total: 3 min_p + 5 top_k + 4 nucleus), 8 iterations each, unattended. All output (checkpoints, metrics, logs) goes to Google Drive so it survives a session reset, and every run's metrics also go to W&B.

Cell 7 (last cell) is set to run **one arm** (`ARM_NAME`) by default -- good for a first full top-to-bottom validation run. Set `ARM_NAME = None` there to run the full 12-arm sweep instead.

If Colab disconnects mid-run: just re-run the notebook top to bottom. Everything already completed (per-iteration and per-arm) is detected and skipped, not redone -- see `main.py`'s `is_generation_complete()` and `run_all_arms.py`'s `is_arm_complete()`.

Run the cells in order, top to bottom. There is nothing else to edit here besides `ARM_NAME` -- all experiment parameters live in `config/arms.yaml` in the repo.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/FinalProject'
assert __import__('os').path.isdir(DRIVE_ROOT), (
    f"Expected Drive folder not found: {DRIVE_ROOT}."
)
print(f"Drive mounted, using {DRIVE_ROOT}")

## 2. Clone this repo

Set `REPO_URL` once -- this is the single documented manual edit point for this notebook.

In [ ]:
REPO_URL = "https://github.com/<your-username>/Thesis-Truncation-Blind-Spot.git"

!git clone {REPO_URL} Thesis-Truncation-Blind-Spot
%cd Thesis-Truncation-Blind-Spot

## 3. Install Colab-specific dependencies

Uses `requirements-colab.txt` (omits `torch` -- Colab preinstalls its own CUDA-matched build).

In [ ]:
!pip install -q git+https://github.com/huggingface/transformers
!pip install -q -r requirements-colab.txt
!pip install -q pyyaml

## 4. Log into Weights & Biases

Store your API key as a Colab secret named `WANDB_API_KEY` (the key icon in the left sidebar) so this runs non-interactively; falls back to an interactive prompt if the secret isn't set.

In [ ]:
import wandb

try:
    from google.colab import userdata
    wandb.login(key=userdata.get('WANDB_API_KEY'))
except Exception:
    wandb.login()

## 5. Environment snapshot

If this does not report a real CUDA device, stop here and check that the Colab runtime type is set to a GPU before continuing.

In [ ]:
!python src/print_env.py --save {DRIVE_ROOT}/logs/env_snapshot_colab.json

## 6. WikiText-103 data

Loads with a 2,000-row cap (`config/dataset/wikitext103.yaml`'s `max_train_samples`) at `DATASET_PATH`. If a previous run already put a *different*-sized `train.json` there (e.g. from the earlier 20,000-row cap), delete that folder on Drive first -- this cell only skips reloading if `train.json` already exists, it doesn't check whether it matches the current cap.

In [ ]:
import os

DATASET_PATH = f"{DRIVE_ROOT}/data/wikitext103"
if not os.path.exists(f"{DATASET_PATH}/train.json"):
    print(f"Not found at {DATASET_PATH}, loading fresh (max_train_samples=2000)...")
    !python src/load_data_ours.py --config-name=ours dataset.path={DATASET_PATH}
else:
    print(f"Reusing existing WikiText-103 data at {DATASET_PATH}")

## 7. Run arm(s)

Reads `config/arms.yaml`: 8 iterations, 30% synthetic contamination per iteration (`human_data_alpha`/`ai_beta` derived from that target -- see the manifest's comments). Writes checkpoints/metrics under `{DRIVE_ROOT}/experiments/<arm_name>/` and logs every run to W&B.

`ARM_NAME` below runs just that one arm end-to-end (today's plan: validate the full pipeline on one arm first). Set `ARM_NAME = None` to run the full 12-arm sweep instead once that's confirmed working.

Safe to re-run unmodified after any disconnect -- already-completed arms and iterations are skipped automatically.

In [ ]:
ARM_NAME = "minp_005"  # set to None to run the full 12-arm sweep

arm_flag = f"--arms {ARM_NAME}" if ARM_NAME else ""
!DRIVE_ROOT={DRIVE_ROOT} python run_all_arms.py --drive-root {DRIVE_ROOT} {arm_flag}